# Bobby Carrot — PPO Reinforcement Learning

**Algorithm**: PPO (Proximal Policy Optimization) with CNN feature extractor  
**Training set**: Normal levels 1–25 (5-stage curriculum)  
**Test set**: Normal levels 26–30 (zero-shot evaluation)  

## Curriculum stages
| Stage | Levels | New mechanic |
|-------|--------|--------------|
| 1 | L01–L03 | Walls, floor, carrots |
| 2 | L01–L07 | + Crumble tiles |
| 3 | L01–L12 | + Directional conveyors |
| 4 | L01–L17 | + Red/Yellow switches |
| 5 | L01–L25 | + Keys/Locks, full mix |

## Before running
1. **Runtime → Change runtime type → T4 GPU** (required)
2. Run cells top-to-bottom. Long-running cells show a progress bar.
3. Models are saved to Google Drive so they survive session disconnects.


In [ ]:
# ── Mount Google Drive (for checkpoint persistence) ──────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = "/content/drive/MyDrive/bobby_carrot_rl"
os.makedirs(f"{DRIVE_DIR}/models", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/tb_logs", exist_ok=True)
print(f"Checkpoints will be saved to: {DRIVE_DIR}")


In [ ]:
# ── GPU + RAM check ───────────────────────────────────────────────────────────
import torch, os

# psutil may not be pre-installed on all Colab images
try:
    import psutil
    cpu_ram_gb = psutil.virtual_memory().total / 1e9
    print(f"CPU RAM  : {cpu_ram_gb:.1f} GB")
except ImportError:
    cpu_ram_gb = 12.0   # Colab free-tier default
    print("CPU RAM  : ~12 GB (psutil not found, assuming free tier)")

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU      : {gpu}")
    print(f"VRAM     : {vram:.1f} GB")
else:
    vram = 0.0
    print("WARNING: No GPU. Runtime → Change runtime type → T4 GPU")

# ── Conservative N_ENVS to avoid CPU RAM exhaustion ───────────────────────────
# Each SubprocVecEnv worker is a full Python process (~200 MB each).
# Colab free tier has ~12 GB CPU RAM. With 16 workers that's ~3 GB for envs
# alone, leaving headroom for Python, model weights, and rollout buffers.
#
# If you have a Colab Pro+ instance with 52 GB RAM, raise to 32.
if cpu_ram_gb >= 50:
    N_ENVS = 32
elif cpu_ram_gb >= 25:
    N_ENVS = 16
else:
    N_ENVS = 8    # safe default for 12 GB free tier

print(f"N_ENVS   : {N_ENVS}  (raise to 16–32 only on high-RAM Colab Pro+ instances)")

In [ ]:
# ── Clone repo + install ──────────────────────────────────────────────────────
import os, subprocess, sys

REPO_URL = "https://github.com/Charan20510/Bobby_Carrot_Python.git"
REPO_DIR = "/content/bobby_carrot"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned, pulling latest...")
    !cd {REPO_DIR} && git pull

# Install the package
!pip install -e {REPO_DIR} -q

# sb3-contrib provides RecurrentPPO (LSTM-based PPO) for stages 3-5
!pip install "stable-baselines3[extra]>=2.3" "sb3-contrib>=2.3" tensorboard tqdm -q

# Force python to recognize the new install without restarting runtime
import site
from importlib import reload
site.main()

print("\nInstall complete.")

In [ ]:
# ── Suppress pygame display (headless Colab) ──────────────────────────────────
import os, sys
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"

# Add REPO_DIR to path explicitly to ensure imports work
sys.path.append("/content/bobby_carrot")

# ── Sanity-check the environment ──────────────────────────────────────────────
import numpy as np
from bobby_carrot.gym_env import BobbyCarrotEnv

env = BobbyCarrotEnv(map_kind="normal", map_number=1)
obs, info = env.reset()

print("Observation keys :", list(obs.keys()))
print("tiles            :", obs["tiles"].shape, obs["tiles"].dtype)
print("player position  : x=%d  y=%d" % (obs["player_x"], obs["player_y"]))
print("carrots          : %d / %d" % (obs["carrot_count"], obs["carrot_total"]))
print("keys (g/y/r)     :", obs["keys"])

obs, r, term, trunc, info = env.step(1)   # move UP
print("\nAfter one step   : reward=%.3f  terminated=%s" % (r, term))
print("Events           :", info["events"])
env.close()

print("\n✓ Environment sanity check passed")

---
## Components
The next two cells define the curriculum wrapper and the CNN feature extractor.  
**Run them once** before starting training.


In [ ]:
import numpy as np
import gymnasium as gym
from bobby_carrot.gym_env import BobbyCarrotEnv

# ── Level pools per curriculum stage ─────────────────────────────────────────
STAGE_ALL_LEVELS = {
    1: list(range(1,  4)),   # L01–L03: walls + floor + carrots
    2: list(range(1,  8)),   # L01–L07: + crumble tiles
    3: list(range(1, 13)),   # L01–L12: + directional conveyors
    4: list(range(1, 18)),   # L01–L17: + global switches
    5: list(range(1, 26)),   # L01–L25: + keys/locks, full mix
}
STAGE_NEW_START = {1: 1, 2: 4, 3: 8, 4: 13, 5: 18}

# Win-rate thresholds for adaptive stage promotion (on hardest level of stage)
STAGE_WIN_THRESHOLD = {1: 0.70, 2: 0.65, 3: 0.55, 4: 0.50, 5: 0.45}
# Minimum and maximum steps per stage regardless of win rate
STAGE_MIN_STEPS = {1: 1_000_000, 2: 2_000_000, 3: 3_000_000, 4: 4_000_000, 5: 5_000_000}
STAGE_MAX_STEPS = {1: 5_000_000, 2: 8_000_000, 3: 12_000_000, 4: 15_000_000, 5: 20_000_000}


# ── Potential function for reward shaping ────────────────────────────────────
def _compute_potential(gs) -> float:
    """Negative normalised Manhattan distance to nearest uncollected carrot.

    When all carrots are collected, targets the exit tile instead.
    Potential is in [-1, 0]; closer to goal → closer to 0.
    Guaranteed not to change the optimal policy (Ng et al., 1999).
    """
    tiles = gs.tiles
    px, py = gs.coord_src

    # Find uncollected carrots (tile 19) or eggs (tile 45)
    targets = [(i % 16, i // 16)
               for i, t in enumerate(tiles) if t in (19, 45)]

    if not targets:
        # All collectibles gathered — guide toward exit
        exit_idx = next((i for i, t in enumerate(tiles) if t == 44), None)
        if exit_idx is not None:
            targets = [(exit_idx % 16, exit_idx // 16)]

    if not targets:
        return 0.0

    min_dist = min(abs(px - cx) + abs(py - cy) for cx, cy in targets)
    return -min_dist / 32.0   # normalise to [-1, 0]


class RewardShapingWrapper(gym.Wrapper):
    """Potential-based reward shaping + progress and efficiency bonuses.

    Shaped reward on each step:
        r'(s,a,s') = r(s,a,s') + γ·Φ(s') - Φ(s)   (Ng et al. 1999)

    Extra bonuses (do not break optimality in a practical sense):
        +key_pickup:        +0.3  (encourages key/lock sequencing)
        +progress_carrot:   +0.1 × (new_count / total)  (dense partial progress)
        +crumble_survived:  +0.2  (reward successfully traversing a crumble tile)
        +efficiency:        +2.0 × (1 - steps/max_steps)  on win
    """
    GAMMA = 0.995

    def __init__(self, env, max_episode_steps: int = 500):
        super().__init__(env)
        self._max_steps = max_episode_steps
        self._prev_potential = 0.0
        self._step_count = 0

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self._step_count = 0
        # Compute initial potential from raw GameState
        gs = self._get_gs()
        self._prev_potential = _compute_potential(gs) if gs else 0.0
        return obs, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self._step_count += 1

        gs = self._get_gs()
        curr_potential = _compute_potential(gs) if gs else 0.0

        # Potential-based shaping
        reward += self.GAMMA * curr_potential - self._prev_potential
        self._prev_potential = curr_potential

        events = info.get("events", [])
        for e in events:
            if e == "key_picked":
                reward += 0.3
            elif e == "carrot_collected":
                # Progress bonus scales with how far along collection is
                gs2 = gs
                if gs2:
                    total = max(gs2.carrot_total + gs2.egg_total, 1)
                    collected = gs2.carrot_count + gs2.egg_count
                    reward += 0.1 * (collected / total)
            elif e == "crumble_survived":   # custom event we'll emit below
                reward += 0.2

        # Efficiency bonus on win
        if terminated and reward > 5.0:   # heuristic: won if reward jumped
            reward += 2.0 * (1.0 - self._step_count / self._max_steps)

        return obs, reward, terminated, truncated, info

    def _get_gs(self):
        """Unwrap to the innermost GameEnv and return its GameState."""
        env = self.env
        while hasattr(env, "env"):
            env = env.env
        return getattr(getattr(env, "_env", None), "gs", None)


class CurriculumEnv(gym.Env):
    """Curriculum-based level sampling with inverse-win-rate weighting.

    Level sampling:
      stage 1 : uniform over stage levels
      stage k>1: weighted by (1 - rolling_win_rate) per level;
                 70% new levels, 30% replay from prior stages
    """
    metadata = {"render_modes": [None]}

    def __init__(self, stage: int = 1, max_episode_steps: int = 500):
        super().__init__()
        assert stage in STAGE_ALL_LEVELS, f"Invalid stage {stage}"
        self.stage = stage
        self.max_episode_steps = max_episode_steps

        new_start = STAGE_NEW_START[stage]
        all_end   = STAGE_ALL_LEVELS[stage][-1]
        self._new_levels   = list(range(new_start, all_end + 1))
        self._prior_levels = list(range(1, new_start)) if new_start > 1 else []
        self._all_levels   = self._prior_levels + self._new_levels

        # Rolling win-rate tracking: deque of last 100 outcomes per level
        from collections import deque
        self._outcomes: dict = {l: deque(maxlen=100) for l in self._all_levels}

        _dummy = BobbyCarrotEnv(map_kind="normal", map_number=1)
        self.observation_space = _dummy.observation_space
        self.action_space      = _dummy.action_space
        _dummy.close()

        self._env        = None
        self._step_count = 0
        self._cur_level  = None

    def _win_rate(self, level: int) -> float:
        outcomes = self._outcomes[level]
        return float(np.mean(outcomes)) if outcomes else 0.5   # optimistic prior

    def _pick_level(self) -> int:
        # 70/30 split between new and prior levels (same as before)
        if self._prior_levels and np.random.random() < 0.30:
            pool = self._prior_levels
        else:
            pool = self._new_levels

        # Inverse win-rate weights within the chosen pool
        weights = np.array([1.0 - self._win_rate(l) for l in pool])
        weights = np.clip(weights, 0.05, 1.0)   # floor prevents starvation
        weights /= weights.sum()
        return int(np.random.choice(pool, p=weights))

    def record_outcome(self, level: int, won: bool) -> None:
        """Called externally (e.g. from WinRateCallback) to log episode results."""
        if level in self._outcomes:
            self._outcomes[level].append(float(won))

    def reset(self, **kwargs):
        if self._env is not None:
            self._env.close()
        self._cur_level = self._pick_level()
        base = BobbyCarrotEnv(map_kind="normal", map_number=self._cur_level)
        self._env = RewardShapingWrapper(base, max_episode_steps=self.max_episode_steps)
        self._step_count = 0
        return self._env.reset(**kwargs)

    def step(self, action):
        self._step_count += 1
        obs, reward, terminated, truncated, info = self._env.step(action)
        if self._step_count >= self.max_episode_steps:
            truncated = True
        if terminated or truncated:
            won = reward > 5.0   # won if large positive reward
            self.record_outcome(self._cur_level, won)
        return obs, reward, terminated, truncated, info

    def close(self):
        if self._env is not None:
            self._env.close()
            self._env = None


# Functional tests
env = CurriculumEnv(stage=1)
obs, _ = env.reset()
assert obs["tiles"].shape == (256,)
obs, r, term, trunc, info = env.step(env.action_space.sample())
env.close()
print("✓ CurriculumEnv functional test passed")

env2 = CurriculumEnv(stage=5)
obs2, _ = env2.reset()
obs2, r2, *_ = env2.step(1)
env2.close()
print("✓ CurriculumEnv stage-5 functional test passed")
print("✓ RewardShapingWrapper with potential shaping active")

In [ ]:
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import VecEnv
import numpy as np


class WinRateCallback(BaseCallback):
    """Track per-level win rates and handle adaptive stage promotion.

    Every `check_freq` steps:
      1. Evaluates `n_eval_episodes` episodes on each level in the current stage.
      2. Logs win rates to TensorBoard.
      3. Signals stage promotion when win_rate(hardest_level) >= threshold
         AND at least STAGE_MIN_STEPS have been trained.
    """

    def __init__(
        self,
        stage: int,
        eval_levels: list,
        n_eval_episodes: int = 20,
        check_freq: int = 100_000,
        verbose: int = 1,
    ):
        super().__init__(verbose)
        self.stage = stage
        self.eval_levels = eval_levels
        self.hardest_level = max(eval_levels)
        self.n_eval_episodes = n_eval_episodes
        self.check_freq = check_freq
        self.promote = False          # set True when threshold met
        self._steps_at_last_check = 0

    def _on_step(self) -> bool:
        if self.num_timesteps - self._steps_at_last_check < self.check_freq:
            return True
        self._steps_at_last_check = self.num_timesteps

        win_rates = {}
        for level in self.eval_levels:
            wins = 0
            for _ in range(self.n_eval_episodes):
                env = BobbyCarrotEnv(map_kind="normal", map_number=level)
                obs, _ = env.reset()
                total_r = 0.0
                for _ in range(500):
                    action, _ = self.model.predict(obs, deterministic=True)
                    obs, r, term, trunc, _ = env.step(action)
                    total_r += r
                    if term or trunc:
                        break
                env.close()
                if total_r > 5.0:
                    wins += 1
            win_rates[level] = wins / self.n_eval_episodes

        # Log to TensorBoard
        for level, wr in win_rates.items():
            self.logger.record(f"eval/win_rate_L{level:02d}", wr)

        hardest_wr = win_rates.get(self.hardest_level, 0.0)
        threshold  = STAGE_WIN_THRESHOLD[self.stage]
        min_steps  = STAGE_MIN_STEPS[self.stage]

        if self.verbose:
            print(f"\n  [WinRateCallback] stage={self.stage}  "
                  f"L{self.hardest_level:02d}_win={hardest_wr:.0%}  "
                  f"threshold={threshold:.0%}  "
                  f"steps={self.num_timesteps:,}")

        if hardest_wr >= threshold and self.num_timesteps >= min_steps:
            if self.verbose:
                print(f"  → Promotion threshold met! Stopping stage {self.stage}.")
            self.promote = True
            return False   # stops model.learn() early

        # Also stop if we've hit the per-stage maximum
        if self.num_timesteps >= STAGE_MAX_STEPS[self.stage]:
            if self.verbose:
                print(f"  → Max steps ({STAGE_MAX_STEPS[self.stage]:,}) reached for stage {self.stage}.")
            self.promote = True
            return False

        return True


print("✓ WinRateCallback defined")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

class _ResBlock(nn.Module):
    def __init__(self, ch: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1), nn.ReLU(),
            nn.Conv2d(ch, ch, 3, padding=1),
        )
    def forward(self, x):
        return F.relu(x + self.net(x))

class _TileAttention(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int = 4):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        tokens = x.flatten(2).permute(0, 2, 1)
        attn_out, _ = self.attn(tokens, tokens, tokens)
        tokens = self.norm(tokens + attn_out)
        return tokens.permute(0, 2, 1).view(B, C, H, W)

class BobbyExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim: int = 256):
        super().__init__(observation_space, features_dim=features_dim)
        EMBED_DIM = 32
        self.embed = nn.Embedding(64, EMBED_DIM)
        self.cnn1 = nn.Sequential(
            nn.Conv2d(EMBED_DIM, 64, kernel_size=3, padding=1), nn.ReLU(),
            _ResBlock(64),
            nn.AdaptiveAvgPool2d(4),
        )
        self.cnn2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            _ResBlock(128),
        )
        self.attn = _TileAttention(embed_dim=128, num_heads=4)
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(2), nn.Flatten())
        self.scalar_net = nn.Sequential(
            nn.Linear(9, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
        )
        self.fuse = nn.Sequential(nn.Linear(512 + 64, features_dim), nn.ReLU())

    def forward(self, obs: dict) -> torch.Tensor:
        tiles = obs["tiles"].view(-1, 256).round().long().clamp(0, 63)
        B = tiles.size(0)
        e = self.embed(tiles)
        e = e.view(B, 16, 16, -1).permute(0, 3, 1, 2)
        x = self.cnn1(e)
        x = self.cnn2(x)
        x = self.attn(x)
        cnn_out = self.pool(x)

        def _proc_scalar(key, dim):
            val = obs[key]
            if val.dim() > 2:
                val = val.flatten(1)
            if val.shape[1] == dim:
                return val.argmax(1, keepdim=True).float() / float(dim - 1)
            return val.view(B, -1).float()

        px = _proc_scalar("player_x", 16)
        py = _proc_scalar("player_y", 16)
        cc = _proc_scalar("carrot_count", 64)
        ct = _proc_scalar("carrot_total", 64)
        ec = _proc_scalar("egg_count", 64)
        et = _proc_scalar("egg_total", 64)
        keys = obs["keys"].view(B, 3).float()

        scalars = torch.cat([px, py, cc, ct, ec, et, keys], dim=1)
        sc_out = self.scalar_net(scalars)
        return self.fuse(torch.cat([cnn_out, sc_out], dim=1))

---
## Training

The cell below runs all **5 curriculum stages** sequentially.

- **Stages 1–2**: plain PPO (fast, no LSTM overhead)  
- **Stages 3–5**: RecurrentPPO with LSTM (multi-step planning for conveyors, switches, key-locks)

Each stage trains until the hardest level in that stage hits its win-rate threshold, then advances. Stage k+1 warm-starts from stage k's best checkpoint.

**Expected wall-clock times** with `N_ENVS=8` on the Colab free tier:
- T4 (~80–120 K steps/s): ≈ 4–8 h total  
- L4 (~150 K steps/s): ≈ 2–4 h total  

If you have Colab Pro+ (52 GB RAM), set `N_ENVS = 32` in the gpu-check cell to triple throughput.

If the session disconnects, re-run from this cell — already-completed stages are skipped automatically.

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
# Stages 1-2: standard PPO (fast, no LSTM overhead)
# Stages 3-5: RecurrentPPO with LSTM (better multi-step planning)
RECURRENT_FROM_STAGE = 3   # set to 6 to disable RecurrentPPO entirely

N_STEPS    = 256   # shorter rollouts → more frequent updates

# Batch size: keep GPU saturated but don't bloat CPU RAM with rollout buffers.
# Rollout buffer holds N_ENVS × N_STEPS samples on CPU — at 8 envs × 256 = 2048,
# a batch of 512 gives 4 mini-batch passes per update.
BATCH_SIZE = 512

# RecurrentPPO needs LSTM state per env — use same env count as PPO since
# we've already halved N_ENVS for CPU RAM safety.
N_ENVS_RECURRENT = N_ENVS   # same count; LSTM overhead is VRAM, not CPU RAM

# Per-stage entropy coefficient: higher early (explore) → lower late (exploit)
STAGE_ENT_COEF = {1: 0.02, 2: 0.02, 3: 0.015, 4: 0.01, 5: 0.01}

# Shared PPO kwargs (used for both PPO and RecurrentPPO)
PPO_BASE_KWARGS = dict(
    n_steps       = N_STEPS,
    batch_size    = BATCH_SIZE,
    n_epochs      = 10,
    gamma         = 0.995,    # higher discount for long-horizon planning
    gae_lambda    = 0.97,
    clip_range    = 0.2,
    max_grad_norm = 0.5,
)

# RecurrentPPO-specific: LSTM hidden size
RECURRENT_KWARGS = dict(
    **PPO_BASE_KWARGS,
    lstm_hidden_size = 256,
    n_lstm_layers    = 1,
)

POLICY_KWARGS = dict(
    features_extractor_class  = BobbyExtractor,
    features_extractor_kwargs = {"features_dim": 256},
    net_arch                  = [],   # extractor outputs 256-dim; no extra MLP
)

steps_per_rollout = N_ENVS * N_STEPS
print(f"N_ENVS               : {N_ENVS}")
print(f"Steps / rollout      : {steps_per_rollout:,}  ({N_ENVS} envs × {N_STEPS} steps)")
print(f"Batch size           : {BATCH_SIZE}  ({steps_per_rollout // BATCH_SIZE} mini-batches/update)")
print(f"RecurrentPPO from    : stage {RECURRENT_FROM_STAGE}")
print(f"Stage step budgets   : min={STAGE_MIN_STEPS}  max={STAGE_MAX_STEPS}")

In [ ]:
import os, torch
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor
from stable_baselines3.common.callbacks import CheckpointCallback

try:
    from sb3_contrib import RecurrentPPO
    _HAS_RECURRENT = True
except ImportError:
    _HAS_RECURRENT = False
    print("WARNING: sb3-contrib not installed. Falling back to PPO for all stages.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def _use_recurrent(stage: int) -> bool:
    return _HAS_RECURRENT and stage >= RECURRENT_FROM_STAGE


def _policy_name(stage: int) -> str:
    return "MultiInputLstmPolicy" if _use_recurrent(stage) else "MultiInputPolicy"


def make_env_fn(stage: int):
    def _init():
        return CurriculumEnv(stage=stage, max_episode_steps=500)
    return _init


def _lr_schedule(stage: int):
    """Cosine annealing from 3e-4 to 1e-5 over the stage's max steps."""
    max_steps = STAGE_MAX_STEPS[stage]
    def schedule(progress_remaining: float) -> float:
        import math
        frac = 1.0 - progress_remaining    # 0 at start → 1 at end
        return 1e-5 + 0.5 * (3e-4 - 1e-5) * (1 + math.cos(math.pi * frac))
    return schedule


for stage in range(1, 6):
    level_range   = f"L{STAGE_ALL_LEVELS[stage][0]:02d}–L{STAGE_ALL_LEVELS[stage][-1]:02d}"
    use_recurrent = _use_recurrent(stage)
    algo_name     = "RecurrentPPO" if use_recurrent else "PPO"
    n_envs        = N_ENVS_RECURRENT if use_recurrent else N_ENVS

    print(f"\n{'='*65}")
    print(f"  STAGE {stage}/5   Levels {level_range}   [{algo_name}  ×{n_envs} envs]")
    print(f"{'='*65}")

    final_path = f"{DRIVE_DIR}/models/stage_{stage}_final.zip"
    if os.path.exists(final_path):
        print(f"  Already complete ({final_path}). Skipping.")
        continue

    model_dir = f"{DRIVE_DIR}/models/stage_{stage}"
    os.makedirs(model_dir, exist_ok=True)

    # ── Vectorised environments ────────────────────────────────────────────
    vec_env  = VecMonitor(SubprocVecEnv([make_env_fn(stage)] * n_envs))

    # ── Build or warm-start model ──────────────────────────────────────────
    AlgoCls   = RecurrentPPO if use_recurrent else PPO
    algo_kwargs = dict(
        **(RECURRENT_KWARGS if use_recurrent else PPO_BASE_KWARGS),
        ent_coef        = STAGE_ENT_COEF[stage],
        learning_rate   = _lr_schedule(stage),
    )

    if stage == 1:
        model = AlgoCls(
            _policy_name(stage),
            vec_env,
            **algo_kwargs,
            policy_kwargs   = POLICY_KWARGS,
            tensorboard_log = f"{DRIVE_DIR}/tb_logs/",
            device          = DEVICE,
            verbose         = 1,
        )
    else:
        # Look for a usable checkpoint from the previous stage
        prev_candidates = [
            f"{DRIVE_DIR}/models/stage_{stage-1}/best_model.zip",
            f"{DRIVE_DIR}/models/stage_{stage-1}_final.zip",
        ]
        load_path = next((p for p in prev_candidates if os.path.exists(p)), None)

        # Can only warm-start if algorithm class is the same
        prev_recurrent = _use_recurrent(stage - 1)
        if load_path and (prev_recurrent == use_recurrent):
            print(f"  Warm-starting from {load_path}")
            model = AlgoCls.load(
                load_path,
                env             = vec_env,
                tensorboard_log = f"{DRIVE_DIR}/tb_logs/",
                device          = DEVICE,
                verbose         = 1,
            )
            # Update per-stage hyperparams after loading
            model.ent_coef      = STAGE_ENT_COEF[stage]
            model.learning_rate = _lr_schedule(stage)
        else:
            if load_path:
                print(f"  Algorithm switch at stage {stage}: starting fresh.")
            model = AlgoCls(
                _policy_name(stage),
                vec_env,
                **algo_kwargs,
                policy_kwargs   = POLICY_KWARGS,
                tensorboard_log = f"{DRIVE_DIR}/tb_logs/",
                device          = DEVICE,
                verbose         = 1,
            )

    # ── Callbacks ─────────────────────────────────────────────────────────
    win_cb = WinRateCallback(
        stage           = stage,
        eval_levels     = STAGE_ALL_LEVELS[stage],
        n_eval_episodes = 20,
        check_freq      = 100_000,
        verbose         = 1,
    )
    ckpt_cb = CheckpointCallback(
        save_freq   = max(500_000 // n_envs, 1),
        save_path   = model_dir,
        name_prefix = "ckpt",
    )

    # ── Train ─────────────────────────────────────────────────────────────
    model.learn(
        total_timesteps     = STAGE_MAX_STEPS[stage],
        callback            = [win_cb, ckpt_cb],
        reset_num_timesteps = (stage == 1),
        tb_log_name         = f"{algo_name}_stage{stage}",
        progress_bar        = True,
    )

    model.save(f"{DRIVE_DIR}/models/stage_{stage}_final")
    vec_env.close()

print("\n✓ All training stages complete!")

In [ ]:
# ── TensorBoard ───────────────────────────────────────────────────────────────
# Run this cell at any time during or after training to view live charts.
%load_ext tensorboard
%tensorboard --logdir {DRIVE_DIR}/tb_logs


---
## Evaluation — Levels 26–30 (held-out test set)

These levels were **never seen during training**.  
The cell below runs 50 episodes per level with a deterministic policy and reports:
- **Win rate**: episodes where agent collects all carrots and reaches the exit
- **Mean reward**: total discounted reward per episode
- **Mean steps**: episode length for winning trajectories
- **Death rate**: fraction of episodes that ended by falling onto a lethal tile


In [ ]:
import os, numpy as np
from stable_baselines3 import PPO
from bobby_carrot.gym_env import BobbyCarrotEnv

try:
    from sb3_contrib import RecurrentPPO
except ImportError:
    RecurrentPPO = None

# ── Load best available model ─────────────────────────────────────────────────
def _load_best_model(drive_dir):
    candidates = []
    for s in range(5, 0, -1):
        candidates += [
            f"{drive_dir}/models/stage_{s}/best_model.zip",
            f"{drive_dir}/models/stage_{s}_final.zip",
        ]
    path = next((p for p in candidates if os.path.exists(p)), None)
    if path is None:
        raise FileNotFoundError("No trained model found. Complete at least one stage.")
    # Try RecurrentPPO first, fall back to PPO
    if RecurrentPPO:
        try:
            return RecurrentPPO.load(path), path
        except Exception:
            pass
    return PPO.load(path), path

model, model_path = _load_best_model(DRIVE_DIR)
print(f"Loaded : {model_path}")
print(f"Type   : {type(model).__name__}")

is_recurrent = RecurrentPPO is not None and isinstance(model, RecurrentPPO)


# ── Failure mode classifier ───────────────────────────────────────────────────
def _classify(total_r, dead, timed_out, carrot_pct):
    if total_r > 5.0:          return "win"
    if dead:                   return "death"
    if timed_out and carrot_pct > 0.9: return "stuck_near_exit"
    if timed_out:              return "timeout"
    return "unknown"


# ── Evaluation loop ───────────────────────────────────────────────────────────
N_EPISODES = 50
WIN_THRESHOLD = 5.0

print(f"\n{'Level':<8} {'Win%':<8} {'Reward':>9} {'Steps':>8} "
      f"{'Death%':>8} {'Timeout%':>10} {'Carrot%':>9}")
print("-" * 65)

all_results = {}
heatmaps    = {}

for level in range(26, 31):
    wins = deaths = timeouts = stuck = 0
    total_r = total_steps = total_carrot_pct = 0.0
    heatmap = np.zeros((16, 16), dtype=float)

    for ep in range(N_EPISODES):
        env  = BobbyCarrotEnv(map_kind="normal", map_number=level)
        obs, _ = env.reset()
        ep_r = ep_steps = 0.0
        dead = timed_out = False

        # RecurrentPPO needs LSTM state threading
        lstm_states = None
        episode_start = np.array([True])

        while True:
            # Visitation heatmap
            heatmap[obs["player_y"]][obs["player_x"]] += 1

            if is_recurrent:
                action, lstm_states = model.predict(
                    obs, state=lstm_states,
                    episode_start=episode_start,
                    deterministic=True,
                )
                episode_start = np.array([False])
            else:
                action, _ = model.predict(obs, deterministic=True)

            obs, r, term, trunc, info = env.step(action)
            ep_r    += r
            ep_steps += 1

            if term:
                dead = (ep_r < WIN_THRESHOLD)
                break
            if trunc or ep_steps >= 500:
                timed_out = True
                break

        env_inner = env._env if hasattr(env, "_env") else None
        gs = getattr(env_inner, "gs", None)
        if gs:
            total_ct = max(gs.carrot_total + gs.egg_total, 1)
            collected = gs.carrot_count + gs.egg_count
            carrot_pct = collected / total_ct
        else:
            carrot_pct = 0.0

        mode = _classify(ep_r, dead, timed_out, carrot_pct)
        if mode == "win":      wins += 1
        elif mode == "death":  deaths += 1
        elif mode in ("timeout", "stuck_near_exit"): timeouts += 1

        total_r          += ep_r
        total_steps      += ep_steps
        total_carrot_pct += carrot_pct
        env.close()

    win_rate    = wins     / N_EPISODES
    death_rate  = deaths   / N_EPISODES
    timeout_rate= timeouts / N_EPISODES
    mean_r      = total_r          / N_EPISODES
    mean_steps  = total_steps      / N_EPISODES
    mean_cpct   = total_carrot_pct / N_EPISODES

    all_results[level] = dict(
        win_rate=win_rate, mean_r=mean_r, mean_steps=mean_steps,
        death_rate=death_rate, timeout_rate=timeout_rate, carrot_pct=mean_cpct,
    )
    heatmaps[level] = heatmap / max(heatmap.sum(), 1)

    print(f"L{level:<7} {win_rate:<8.0%} {mean_r:>9.2f} {mean_steps:>8.1f} "
          f"{death_rate:>8.0%} {timeout_rate:>10.0%} {mean_cpct:>9.0%}")

avg_win = np.mean([v["win_rate"]  for v in all_results.values()])
avg_r   = np.mean([v["mean_r"]    for v in all_results.values()])
avg_cpct= np.mean([v["carrot_pct"]for v in all_results.values()])
print("-" * 65)
print(f"{'Average':<8} {avg_win:<8.0%} {avg_r:>9.2f} {'':>8} "
      f"{'':>8} {'':>10} {avg_cpct:>9.0%}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

levels    = list(all_results.keys())
win_rates = [all_results[l]["win_rate"] * 100 for l in levels]
rewards   = [all_results[l]["mean_r"]          for l in levels]
deaths    = [all_results[l]["death_rate"] * 100 for l in levels]
timeouts  = [all_results[l]["timeout_rate"]* 100 for l in levels]
carrots   = [all_results[l]["carrot_pct"]  * 100 for l in levels]
labels    = [f"L{l}" for l in levels]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# ── Win rate ──────────────────────────────────────────────────────────────────
ax = axes[0, 0]
colors = ["#2ecc71" if w >= 50 else "#e74c3c" for w in win_rates]
ax.bar(labels, win_rates, color=colors, edgecolor="k", linewidth=0.5)
ax.axhline(50, color="grey", linestyle="--", linewidth=1)
ax.set_title("Win Rate (%) — L26–30"); ax.set_ylim(0, 110)
for i, v in enumerate(win_rates):
    ax.text(i, v + 2, f"{v:.0f}%", ha="center", fontsize=9)

# ── Mean reward ───────────────────────────────────────────────────────────────
ax = axes[0, 1]
ax.bar(labels, rewards, color="#3498db", edgecolor="k", linewidth=0.5)
ax.set_title("Mean Episode Reward — L26–30")
for i, v in enumerate(rewards):
    ax.text(i, max(v + 0.2, 0.2), f"{v:.1f}", ha="center", fontsize=9)

# ── Failure mode breakdown ────────────────────────────────────────────────────
ax = axes[0, 2]
wins_arr    = np.array(win_rates)
deaths_arr  = np.array(deaths)
timeout_arr = np.array(timeouts)
other_arr   = 100 - wins_arr - deaths_arr - timeout_arr
x = np.arange(len(labels))
ax.bar(x, wins_arr,    label="Win",     color="#2ecc71")
ax.bar(x, deaths_arr,  label="Death",   color="#e74c3c", bottom=wins_arr)
ax.bar(x, timeout_arr, label="Timeout", color="#f39c12", bottom=wins_arr+deaths_arr)
ax.bar(x, np.clip(other_arr,0,100), label="Other", color="#95a5a6",
       bottom=wins_arr+deaths_arr+timeout_arr)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_title("Failure Mode Breakdown — L26–30")
ax.set_ylim(0, 110); ax.legend(loc="upper right", fontsize=8)

# ── Mean carrot completion % ──────────────────────────────────────────────────
ax = axes[1, 0]
ax.bar(labels, carrots, color="#9b59b6", edgecolor="k", linewidth=0.5)
ax.axhline(100, color="grey", linestyle="--", linewidth=1)
ax.set_title("Mean Carrot Completion % — L26–30"); ax.set_ylim(0, 110)
for i, v in enumerate(carrots):
    ax.text(i, v + 2, f"{v:.0f}%", ha="center", fontsize=9)

# ── Visitation heatmaps (first two unseen levels) ─────────────────────────────
for col, level in enumerate([26, 27]):
    ax = axes[1, col + 1]
    im = ax.imshow(heatmaps[level], cmap="hot", interpolation="nearest",
                   vmin=0, vmax=heatmaps[level].max())
    ax.set_title(f"Visit Density — L{level}")
    ax.set_xlabel("x"); ax.set_ylabel("y")
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
chart_path = f"{DRIVE_DIR}/eval_results_v2.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart saved to {chart_path}")

---
## Optional — Replay a trained agent

The cell below runs one episode on a chosen level and records the tile-grid at each step.  
Requires `matplotlib` only (no pygame display needed).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from bobby_carrot.gym_env import BobbyCarrotEnv

REPLAY_LEVEL = 26   # change to any level 1–30
MAX_STEPS    = 400

env  = BobbyCarrotEnv(map_kind="normal", map_number=REPLAY_LEVEL)
obs, _ = env.reset()

frames       = [obs["tiles"].reshape(16, 16).copy()]
total_reward = 0.0
done         = False

# Thread LSTM state for RecurrentPPO; ignored for plain PPO
lstm_states   = None
episode_start = np.array([True])

for _ in range(MAX_STEPS):
    if is_recurrent:
        action, lstm_states = model.predict(
            obs, state=lstm_states,
            episode_start=episode_start,
            deterministic=True,
        )
        episode_start = np.array([False])
    else:
        action, _ = model.predict(obs, deterministic=True)

    obs, r, term, trunc, info = env.step(action)
    frames.append(obs["tiles"].reshape(16, 16).copy())
    total_reward += r
    if term or trunc:
        done = True
        break
env.close()

outcome = "WON" if total_reward >= 5.0 else "lost"
print(f"Level {REPLAY_LEVEL}: {outcome}  reward={total_reward:.2f}  steps={len(frames)-1}")

# Animate (every 4th frame for speed)
fig, ax = plt.subplots(figsize=(4, 4))
ax.axis("off")
im = ax.imshow(frames[0], cmap="tab20", vmin=0, vmax=46, interpolation="nearest")
title = ax.set_title("Step 0")

def _update(i):
    idx = min(i * 4, len(frames) - 1)
    im.set_data(frames[idx])
    title.set_text(f"Step {idx}")
    return [im, title]

n_frames = (len(frames) + 3) // 4
ani = animation.FuncAnimation(fig, _update, frames=n_frames, interval=120, blit=True)
plt.close()
HTML(ani.to_jshtml())